In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

image_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
train_full = datasets.CIFAR10(root='data', train=True, transform=image_transform, download=True)
test_full = datasets.CIFAR10(root='data', train=False, transform=image_transform, download=True)

TRAIN_LIMIT = 4096
TEST_LIMIT = 1000
train_dataset = Subset(train_full, range(TRAIN_LIMIT))
test_dataset = Subset(test_full, range(TEST_LIMIT))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

class_names = train_full.classes
print("Class names:", class_names)
images, labels = next(iter(train_loader))
print("image shape:", images.shape)
print("labels shape:", labels.shape)

Device: cuda
Class names: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
image shape: torch.Size([64, 3, 32, 32])
labels shape: torch.Size([64])


In [2]:
class ToyModel(nn.Module):
    def __init__(self, num_classes, in_channels=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_channels * 32 * 32, 512),
            nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


toy_model = ToyModel(10)
toy_model.to(DEVICE)

ToyModel(
  (features): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3072, out_features=512, bias=True)
    (2): ReLU()
  )
  (classifier): Sequential(
    (0): Linear(in_features=512, out_features=10, bias=True)
  )
)

In [8]:
images = images.to(DEVICE)
labels = labels.to(DEVICE)
logits = toy_model(images)
print("logits shape:", logits.shape)

logits shape: torch.Size([64, 10])


In [13]:
criterion = nn.CrossEntropyLoss()
pytorch_loss = criterion(logits, labels)
# print(logits)
# print(pytorch_loss)


tensor(2.3084, device='cuda:0', grad_fn=<NllLossBackward0>)


In [20]:
probabilities = torch.softmax(logits, dim=1)
# for i in range(10):
print(labels)
print(probabilities)
probabilities = probabilities[torch.arange(len(labels)), labels]
# print(torch.arange(len(labels)))

tensor([7, 2, 9, 3, 5, 8, 4, 0, 3, 1, 1, 7, 2, 5, 8, 1, 4, 4, 1, 8, 4, 7, 4, 2,
        8, 6, 5, 1, 6, 2, 0, 5, 0, 0, 5, 4, 9, 6, 0, 9, 0, 6, 2, 4, 9, 8, 6, 0,
        5, 2, 1, 7, 8, 8, 7, 8, 3, 1, 1, 9, 3, 3, 8, 4], device='cuda:0')
tensor([[0.0954, 0.1175, 0.0880, 0.0932, 0.0908, 0.1116, 0.1198, 0.0960, 0.0970,
         0.0907],
        [0.0921, 0.0867, 0.0852, 0.0997, 0.1112, 0.1202, 0.0962, 0.1088, 0.0965,
         0.1034],
        [0.0888, 0.1146, 0.0761, 0.1057, 0.0908, 0.1056, 0.1149, 0.0920, 0.1103,
         0.1013],
        [0.0880, 0.1024, 0.0971, 0.0938, 0.0963, 0.1130, 0.1032, 0.1058, 0.0980,
         0.1025],
        [0.1012, 0.1079, 0.0830, 0.1048, 0.0810, 0.0930, 0.1185, 0.1052, 0.1141,
         0.0914],
        [0.0984, 0.1166, 0.0904, 0.1033, 0.0926, 0.1052, 0.1086, 0.0937, 0.1039,
         0.0872],
        [0.1004, 0.1119, 0.0835, 0.0996, 0.0763, 0.1144, 0.1095, 0.1041, 0.1092,
         0.0910],
        [0.0974, 0.1098, 0.1113, 0.1061, 0.0826, 0.1129, 0.0996, 0.0995, 

In [21]:
print(probabilities)

tensor([0.0960, 0.0852, 0.1013, 0.0938, 0.0930, 0.1039, 0.0763, 0.0974, 0.0949,
        0.0982, 0.1008, 0.1099, 0.0925, 0.1069, 0.1193, 0.0876, 0.0950, 0.0852,
        0.0975, 0.0907, 0.0948, 0.1178, 0.0838, 0.0879, 0.0921, 0.1127, 0.1112,
        0.0987, 0.1013, 0.0896, 0.0816, 0.1115, 0.1087, 0.0954, 0.1202, 0.0923,
        0.1048, 0.1189, 0.0901, 0.1001, 0.0828, 0.1065, 0.0964, 0.0828, 0.1229,
        0.1065, 0.1258, 0.0978, 0.1027, 0.0928, 0.1022, 0.1178, 0.0971, 0.0952,
        0.1069, 0.0930, 0.1195, 0.1258, 0.1129, 0.1044, 0.0916, 0.1027, 0.0928,
        0.0878], device='cuda:0', grad_fn=<IndexBackward0>)


In [34]:
import math

print("probabilities: ", probabilities[:1])
print("math log: ", math.log(1 / len(class_names)))
print("loss: ", pytorch_loss.item())

probabilities:  tensor([[0.0954, 0.1175, 0.0880, 0.0932, 0.0908, 0.1116, 0.1198, 0.0960, 0.0970,
         0.0907]], device='cuda:0', grad_fn=<SliceBackward0>)
math log:  -2.3025850929940455
loss:  2.308382511138916
